# FALSE-STABILITY — full analysis pipeline (Google Colab)

Reproduces every analysis in *"Apparent normalization of vital signs and residual risk after emergency department care"* (MIMIC-IV-ED 2.2 + MIMIC-IV 3.1; MC-MED 1.0.0).

> **Data use.** Run this only on your own credentialed copy of the data, stored in your own Google Drive. Do **not** print, paste or upload row-level data, cohort files or trained models anywhere — including AI assistants (PhysioNet data use agreements). Every cell below prints aggregate output only.

**How to use it**
1. Put the source files in one Drive folder as described in `DATA_ACCESS.md`.
2. Run **Section 0** at the start of *every* Colab session (also after switching CPU ↔ GPU).
3. Run the sections in order. Sections 2 and 4 need a **GPU** runtime (*Runtime → Change runtime type → T4 GPU*). Section 6 benefits from a **high-RAM** runtime.
4. All results go to `FOLDER/salida`, `FOLDER/salida_mcmed`, `FOLDER/salida_fase7`, and the final figures to `FOLDER/AEM_envio_final`.

| Section | Runtime | ≈ Time | Produces |
|---|---|---|---|
| 0 Setup | any | 1 min | — |
| 1 MIMIC-IV-ED cohorts | CPU | 15 min | cohorts A/B, Figure 1 counts |
| 2 Exploratory ML, MIMIC | GPU | 70 min | Supplement S5 |
| 3 MC-MED cohorts | CPU | 25 min | cohorts A/B* |
| 4 Transport / local ML | GPU | 50 min | Supplement S6 |
| 5 Final analyses | CPU | 95 min | Tables 1–3, Figures 2–4 |
| 6 Verification V1–V4 | CPU, high RAM | 2 × 20 min | Supplement S4 |
| 7 Final figures | CPU | 1 min | Figures 1–4 (TIFF 600 dpi) |

The exploratory ML (sections 2 and 4) was run **before** the cohort-A outcome correction (step 7e). Keep this order to reproduce the reported results.

## 0 · Setup (run at the start of every session)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# >>> EDIT: your Drive folder with the PhysioNet files (see DATA_ACCESS.md)
FOLDER = "/content/drive/MyDrive/ARTICULOS MIOS/False Stability"
dest = FOLDER + "/mcmed"          # MC-MED folder (visits.csv, numerics.csv, orders.csv)
OUT = FOLDER + "/salida"          # MIMIC-IV-ED derived files

# >>> EDIT: public repository URL. Leave empty to use a copy in FOLDER/codigo/false-stability-ed
REPO_URL = "https://github.com/USUARIO/false-stability-ed"

import os, sys, shutil, subprocess
REPO = "/content/false-stability-ed"
if not os.path.isdir(REPO):
    if REPO_URL and "USUARIO" not in REPO_URL:
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
    else:
        shutil.copytree(os.path.join(FOLDER, "codigo", "false-stability-ed"), REPO)
SRC = os.path.join(REPO, "src")
shutil.copy(os.path.join(SRC, "false_stability_cohort.py"), "/content/")   # needed by the sequence models
sys.path.insert(0, "/content"); sys.path.insert(0, SRC)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-r", os.path.join(REPO, "requirements.txt")], check=True)
print("Code:", SRC)
print("Data folder contents:", sorted(os.listdir(FOLDER)))

## 1 · MIMIC-IV-ED: cohorts and modified NEWS2 (CPU)
Builds cohorts A (discharge home) and B (ward first unit), the vital-sign sets and the exposure groups.

In [ ]:
import zipfile, glob
os.makedirs("/content/mimic/hosp", exist_ok=True); os.makedirs("/content/mimic/icu", exist_ok=True)
zipfile.ZipFile(os.path.join(FOLDER, "mimic-iv-ed-2.2.zip")).extractall("/content/mimic")
for f in ("patients", "admissions", "transfers"):
    shutil.copy(os.path.join(FOLDER, f + ".csv.gz"), "/content/mimic/hosp/")
shutil.copy(os.path.join(FOLDER, "icustays.csv.gz"), "/content/mimic/icu/")

def find_dir(table):
    hits = glob.glob(f"/content/mimic/**/{table}.csv*", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{table}.csv(.gz) not found")
    return os.path.dirname(hits[0])
ED_DIR, HOSP_DIR, ICU_DIR = find_dir("edstays"), find_dir("patients"), find_dir("icustays")
print(ED_DIR, HOSP_DIR, ICU_DIR, sep="\n")

In [ ]:
!python "$SRC/false_stability_cohort.py" --ed "$ED_DIR" --hosp "$HOSP_DIR" --icu "$ICU_DIR" --out "$OUT"

In [ ]:
import pandas as pd
print(pd.read_csv(os.path.join(OUT, "flujo_seleccion.csv")).to_string(index=False)); print()
print(pd.read_csv(os.path.join(OUT, "resumen_fenomeno.csv")).to_string(index=False))

Step 2b — first adjusted analysis (historical; superseded by 7a).

In [ ]:
%run -i "$SRC/fase2b_ajustado.py"

## 2 · Exploratory machine learning in MIMIC-IV-ED (GPU)
Switch to a GPU runtime, run **Section 0**, then these cells **in the same session** (each step uses objects created by the previous one).

In [ ]:
%run -i "$SRC/fase3_lineas_base.py"        # M0–M2 baselines

In [ ]:
%run -i "$SRC/fase4_interpretabilidad.py"  # ablations + TreeSHAP

In [ ]:
%run -i "$SRC/fase5_red_secuencia.py"      # M3h: GRU + FT-Transformer

In [ ]:
%run -i "$SRC/fase5b_robustez.py"          # timing channels, 5 seeds, recalibration

In [ ]:
%run -i "$SRC/fase5c_solo_ritmo.py"        # "rhythm-only" ablation

If the session was interrupted, `fase5c_independiente.py` reruns steps 3 + 5 + 5b + 5c on its own (Section 0 first):

In [ ]:
# %run -i "$SRC/fase5c_independiente.py"

## 3 · MC-MED: external replication of the phenomenon (CPU)
Same frozen definitions as MIMIC-IV-ED. Cohort A (discharge) and B* (Inpatient/Observation).

In [ ]:
%run -i "$SRC/fase6a_explorar_mcmed.py"   # aggregate exploration

In [ ]:
%run -i "$SRC/fase6b_mcmed_fenomeno.py"   # cohorts A and B*

## 4 · Transport and local retraining (GPU)
GPU runtime → Section 0 → these cells. Each is self-contained.

In [ ]:
%run -i "$SRC/fase6c_transporte.py"       # MIMIC-trained models applied unchanged to MC-MED

In [ ]:
%run -i "$SRC/fase6d_local_mcmed.py"      # models retrained within MC-MED

## 5 · Final analyses reported in the main text (CPU)
**7e** rewrites cohort A with the final outcome (any ED return ≤72 h ending in admission). It backs up the previous files and **refuses to run twice**. Then 7a → 7b → 7c-1 → 7c-2.

In [ ]:
%run -i "$SRC/fase7e_desenlace_A_corregido.py"   # run ONCE

In [ ]:
%run -i "$SRC/fase7a_epidemiologia.py"           # Table 2, Figure 2: clustered logistic GLM, M1–M3, interaction

In [ ]:
%run -i "$SRC/fase7b_news2_incremental.py"       # Table 3, Figure 3: N0–N3, grouped CV + temporal split

In [ ]:
%run -i "$SRC/fase7c_sensibilidad_calibracion.py"  # N4, temporal calibration, relative CITL, bootstrap, Table 1

In [ ]:
%run -i "$SRC/fase7c_tablas_consolidadas.py"     # consolidated + supplementary tables, trazabilidad.csv

## 6 · Verification analyses V1–V4 (CPU, high-RAM runtime)
Run one center per session to limit memory use.

In [ ]:
PARTE = "mimic"
%run -i "$SRC/fase7d_verificacion.py"

In [ ]:
PARTE = "mcmed"
%run -i "$SRC/fase7d_verificacion.py"

## 7 · Final figures for submission (CPU)
Writes Figure1–4 (.tif 600 dpi and .png) to `FOLDER/AEM_envio_final`.

In [ ]:
%run -i "$SRC/fase7g_figuras_drive.py"

## Before sharing anything
Only `salida_fase7/*.csv`, the figures and console output are aggregate. `cohorte_*.parquet`, `mcmed_*.csv.gz` and anything in `salida/`, `salida_mcmed/` or `mcmed/` are patient-level and must stay in your credentialed storage.